In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from asyncssh import forward
from sklearn.model_selection import train_test_split
from torch import optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import re
import copy
from sklearn.decomposition import PCA
import seaborn as sns
import torch.nn.functional as F

In [45]:
data=pd.read_csv('../dhi_medo_2c_30n/dihedrals_medoids_new_bb.csv')

# tokens - top_time model
vocab_top = ['-p', '+p', '-l', '+l', 'd']
vocab_seq = ['1', '2', '3', '4']
vocab = {}
token_id = 0

for t in vocab_top:
    for s in vocab_seq:
        fused = f'{t}_{s}'
        vocab[fused] = token_id
        token_id += 1


def tokenize_sequence(token, vocab):
    tokenized = []
    for i in range(3):  # Since sequences are exactly length 3
        fused_key = token[i]
        tokenized.append(vocab[fused_key])
    return tokenized



# Tokenize data
data['splited_top'] = data['top'].apply(lambda text: re.findall(r'[+-][pl]|d', str(text)))
data['splited_seq'] = data['seq'].apply(lambda text: re.findall(r'1|2|3|4', str(text)))

data['tokens'] = data.apply(lambda row: [f'{top}_{seq}' for top, seq in zip(row['splited_top'], row['splited_seq'])],
                            axis=1)
data['tokenased'] = data.apply(lambda row: tokenize_sequence(row['tokens'], vocab), axis=1)
data=data.drop(['top','seq','tokens', 'splited_top', 'splited_seq','lay','copy','time','cluster_id','rank','og_idx'], axis=1)
#data=data[data['sredni_czas']<1000000]
#data=data[data['sredni_czas']>0]
print(data)


           b1     g1      d1      e1      z1      a2      b2     g2      d2  \
0     -112.45  89.19  147.52  174.46  -86.33 -103.99 -161.53  51.25  135.41   
1     -131.98  63.45  141.31  177.71  -84.94  -64.48  176.59  43.75  132.43   
2     -139.35  77.34  137.55 -165.87  -90.25  -73.61  162.06  47.89  135.92   
3     -126.16  76.12  140.29 -163.58  -89.85  -71.27  172.88  41.97  135.93   
4     -132.68  81.89  142.41  171.67  -72.38  -81.81  164.75  50.17  128.81   
...       ...    ...     ...     ...     ...     ...     ...    ...     ...   
74959 -149.11  74.15  139.81 -156.21  -29.11 -157.35  149.63  66.28  148.74   
74960 -131.48  71.21   85.52 -127.30 -109.47  -68.41  154.21  57.68  147.37   
74961 -149.11  74.15  139.81 -156.21  -29.11 -157.35  149.63  66.28  148.74   
74962 -131.48  71.21   85.52 -127.30 -109.47  -68.41  154.21  57.68  147.37   
74963 -127.72 -30.92  131.42 -157.63  -71.09  -79.44  154.64  70.09  121.77   

           e2  ...  g19  d19  e19  z19  a20  b20  g

In [48]:
cols_x=[c for c in data.columns if c not in ['tokenased']]
data_rad=np.radians(data[cols_x])
data_sin=np.sin(data_rad).add_suffix('_sin')
data_cos=np.cos(data_rad).add_suffix('_cos')
data_sincos=pd.concat([data_sin, data_cos], axis=1)
interleaved_cols = [f"{col}_{trig}" for col in cols_x for trig in ["sin", "cos"]]
data_sincos = data_sincos[interleaved_cols]
cols_x=[c for c in data_sincos.columns if c not in ['tokenased']]
data_sincos=pd.concat([data_sincos,data['tokenased']], axis=1)
data_sincos=data_sincos.fillna(0)

X = data_sincos[cols_x]
Y = data_sincos['tokenased']


# Split data
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.1, random_state=42, stratify=data_sincos['tokenased'])


# X tensors: Convert pandas series of lists -> list of lists -> LongTensor (required for nn.Embedding)
X_train_tensor = torch.tensor(np.vstack(X_train.values).astype(np.float32), dtype=torch.float32)
X_test_tensor = torch.tensor(np.vstack(X_test.values).astype(np.float32), dtype=torch.float32)

# Y tensors: Extract values first, then convert to FloatTensor
Y_train_tensor = torch.tensor(Y_train.tolist(), dtype=torch.long)
Y_test_tensor = torch.tensor(Y_test.tolist(), dtype=torch.long)

# Datasets and Loaders
batch_size = 32
train_dataset = TensorDataset(X_train_tensor, Y_train_tensor)

test_dataset = TensorDataset(X_test_tensor, Y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)


In [49]:
print(X_train.shape, X_test.shape)

(67467, 236) (7497, 236)


In [51]:
class StructPred(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_tokens, num_out=3):
        super(StructPred, self).__init__()


        self.num_tokens = num_tokens
        self.num_out = num_out


        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, 32)
        self.fc3 = nn.Linear(32, num_tokens * num_out)

    def forward(self, x):
        # Pass through the network
        x = F.relu(self.fc1(x))
        x = F.dropout(x, p=0.2, training=self.training)
        x = F.relu(self.fc2(x))
        output = self.fc3(x)
        output = output.view(-1, self.num_tokens, self.num_out)

        return output

model = StructPred(input_dim=236, hidden_dim=64, num_tokens=20 ,num_out=3)
optimizer = optim.Adam(model.parameters(), lr=0.00001, weight_decay=2.25e-4)
criterion = nn.CrossEntropyLoss()

EPOCHS = 10000
best_loss = float('inf')
how_many=25
counter_dd=0
best_so_far=None

for epoch in range(EPOCHS):
    model.train() # Set to training mode
    total_loss = 0

    for batch_X, batch_Y in train_loader:
        # 1. Reset gradients
        optimizer.zero_grad()

        # 2. Forward pass: get predictions
        predictions = model(batch_X)

        # 3. Calculate how wrong the predictions were
        loss = criterion(predictions, batch_Y)

        # 4. Backward pass: calculate gradients and update weights
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    # Print progress
    avg_loss = total_loss / len(train_loader)
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS} | Training Loss: {avg_loss:.4f}")

    if total_loss < best_loss:
        best_loss = total_loss
        counter_dd = 0
        best_so_far=copy.deepcopy(model.state_dict())
    else:
        counter_dd += 1

    if counter_dd >= how_many:
        print("Early Stopping")
        break

print("Training Complete!")

Epoch 5/10000 | Training Loss: 2.7563
Epoch 10/10000 | Training Loss: 2.5123
Epoch 15/10000 | Training Loss: 2.3382
Epoch 20/10000 | Training Loss: 2.2218
Epoch 25/10000 | Training Loss: 2.1373
Epoch 30/10000 | Training Loss: 2.0717
Epoch 35/10000 | Training Loss: 2.0129
Epoch 40/10000 | Training Loss: 1.9634
Epoch 45/10000 | Training Loss: 1.9175
Epoch 50/10000 | Training Loss: 1.8775
Epoch 55/10000 | Training Loss: 1.8386
Epoch 60/10000 | Training Loss: 1.8062
Epoch 65/10000 | Training Loss: 1.7750
Epoch 70/10000 | Training Loss: 1.7474
Epoch 75/10000 | Training Loss: 1.7213
Epoch 80/10000 | Training Loss: 1.6989
Epoch 85/10000 | Training Loss: 1.6758
Epoch 90/10000 | Training Loss: 1.6548
Epoch 95/10000 | Training Loss: 1.6359
Epoch 100/10000 | Training Loss: 1.6177
Epoch 105/10000 | Training Loss: 1.5967
Epoch 110/10000 | Training Loss: 1.5832
Epoch 115/10000 | Training Loss: 1.5648
Epoch 120/10000 | Training Loss: 1.5475
Epoch 125/10000 | Training Loss: 1.5355
Epoch 130/10000 | Tr

In [52]:
model.load_state_dict(best_so_far)

# 1. Put the model in evaluation mode
model.eval()

# 2. Turn off gradient tracking for testing (saves memory and speeds up math)
with torch.no_grad():
    # Pass the entire test set through the model at once
    test_logits = model(X_test_tensor)  # Shape: [Batch Size, 20, 3]

    # 3. Get the predicted token IDs!
    # dim=1 tells PyTorch to look across the 20 classes and pick the highest probability.
    # This crushes the shape down to [Batch Size, 3]
    predicted_tokens = torch.argmax(test_logits, dim=1)

    # ==========================================
    # 4. Calculate Accuracies
    # ==========================================
    total_rows = len(Y_test_tensor)
    total_tokens = total_rows * 3

    # Token-Level Accuracy (How many individual tokens are correct?)
    correct_tokens = (predicted_tokens == Y_test_tensor).sum().item()
    token_accuracy = (correct_tokens / total_tokens) * 100

    # Sequence-Level Accuracy (How many rows got ALL 3 tokens exactly right?)
    # .all(dim=1) ensures we only count a row as correct if token1 AND token2 AND token3 match.
    correct_sequences = (predicted_tokens == Y_test_tensor).all(dim=1).sum().item()
    sequence_accuracy = (correct_sequences / total_rows) * 100

print(f"Token-Level Accuracy:    {token_accuracy:.2f}%")
print(f"Sequence-Level Accuracy: {sequence_accuracy:.2f}%")
print("-" * 40)

# ==========================================
# 5. Let's look at 5 actual examples!
# ==========================================
print("Sample Predictions vs Actual:")

n_samples = 50

# Grab the first 5 rows
sample_preds = predicted_tokens[:n_samples].numpy()
sample_actuals = Y_test_tensor[:n_samples].numpy()

for i in range(n_samples):
    # Check if the entire sequence of 3 matches
    if (sample_preds[i] == sample_actuals[i]).all():
        match_status = "✅ Perfect Sequence!"
    else:
        # Check how many tokens matched in this row
        matches = (sample_preds[i] == sample_actuals[i]).sum()
        match_status = f"⚠️ {matches}/3 tokens match"

    print(f"Row {i+1}:")
    print(f"  Predicted: {sample_preds[i]}")
    print(f"  Actual:    {sample_actuals[i]} {match_status}\n")

Token-Level Accuracy:    83.98%
Sequence-Level Accuracy: 61.69%
----------------------------------------
Sample Predictions vs Actual:
Row 1:
  Predicted: [ 7 18 13]
  Actual:    [ 7 18  9] ⚠️ 2/3 tokens match

Row 2:
  Predicted: [ 7 12  7]
  Actual:    [ 7 12  7] ✅ Perfect Sequence!

Row 3:
  Predicted: [14  5  5]
  Actual:    [14  5  5] ✅ Perfect Sequence!

Row 4:
  Predicted: [10 10  2]
  Actual:    [10 10  2] ✅ Perfect Sequence!

Row 5:
  Predicted: [ 9 11 11]
  Actual:    [ 9 11 11] ✅ Perfect Sequence!

Row 6:
  Predicted: [13  4  5]
  Actual:    [13  5  5] ⚠️ 2/3 tokens match

Row 7:
  Predicted: [ 2 11  9]
  Actual:    [ 2 11  9] ✅ Perfect Sequence!

Row 8:
  Predicted: [3 1 2]
  Actual:    [3 1 2] ✅ Perfect Sequence!

Row 9:
  Predicted: [11 18  5]
  Actual:    [11 18  5] ✅ Perfect Sequence!

Row 10:
  Predicted: [12  4 15]
  Actual:    [12  4 15] ✅ Perfect Sequence!

Row 11:
  Predicted: [ 0 10  1]
  Actual:    [ 0 10  1] ✅ Perfect Sequence!

Row 12:
  Predicted: [ 1 11  0]
 

In [8]:
print(X_test, Y_test)

         b1_sin    b1_cos    g1_sin    g1_cos    d1_sin    d1_cos    e1_sin  \
73434 -0.720309 -0.693653  0.982515  0.186181  0.836955 -0.547271 -0.742561   
69852 -0.780430 -0.625243  0.955639  0.294541  0.422460 -0.906382 -0.443853   
9803  -0.933205 -0.359345  0.999379  0.035248  0.726695 -0.686961 -0.332984   
70611 -0.815936 -0.578142  0.930033  0.367475  0.607653 -0.794203 -0.285688   
52422 -0.471935 -0.881633  0.936305  0.351188  0.593981 -0.804479 -0.141247   
...         ...       ...       ...       ...       ...       ...       ...   
2541  -0.616074 -0.787688  0.982839  0.184466  0.659608 -0.751610 -0.199026   
54730 -0.661050 -0.750342  0.980921  0.194406  0.442445 -0.896796 -0.242769   
12802 -0.787043 -0.616899  0.988938  0.148327  0.542002 -0.840377  0.008901   
34613 -0.889656 -0.456632  0.974291  0.225291  0.711045 -0.703147 -0.284685   
71018 -0.841888 -0.539653  0.996507  0.083504  0.673658 -0.739043 -0.018848   

         e1_cos    z1_sin    z1_cos  ...  a20_sin  

In [13]:
print(vocab)

{'-p_1': 0, '-p_2': 1, '-p_3': 2, '-p_4': 3, '+p_1': 4, '+p_2': 5, '+p_3': 6, '+p_4': 7, '-l_1': 8, '-l_2': 9, '-l_3': 10, '-l_4': 11, '+l_1': 12, '+l_2': 13, '+l_3': 14, '+l_4': 15, 'd_1': 16, 'd_2': 17, 'd_3': 18, 'd_4': 19}


In [53]:
torch.save(model.state_dict(), '../modele - wytrenowane/angel_to_struct.pth')